In [15]:
import pandas as pd # importando biblioteca essencial

In [16]:
df = pd.read_excel('../data/data.xlsx') # lendo o arquivo excel e armazenando em um DataFrame

In [17]:
df.info() # exibindo informações sobre o DataFrame, como número de linhas, colunas e tipos de dados

<class 'pandas.DataFrame'>
RangeIndex: 452 entries, 0 to 451
Data columns (total 20 columns):
 #   Column                                              Non-Null Count  Dtype  
---  ------                                              --------------  -----  
 0   Município de vacinação                              444 non-null    str    
 1   Idade Evento                                        444 non-null    float64
 2   Data da Notificação                                 444 non-null    object 
 3   Raça/Cor                                            444 non-null    object 
 4   Sexo                                                444 non-null    str    
 5   Imunobiológico (vacina)                             444 non-null    str    
 6   Dose                                                443 non-null    str    
 7   Gestante                                            444 non-null    object 
 8   Mês de gestação                                     444 non-null    float64
 9   Mulher ama

# Tratamento da coluna data

A coluna data não segue uma padronização clara. Enquanto algumas linhas apresentam apenas a informação do ano, outras apresentam a data completa.

É necessário tratar e padronizar a coluna de data.

In [18]:
# Convertendo a coluna 'Data da Notificação' para string e removendo espaços em branco
df['Data da Notificação'] = df['Data da Notificação'].astype(str).str.strip()

In [19]:
# Extraindo o ano da notificação e criando uma nova coluna 'Ano_Notificacao'
df['Ano_Notificacao'] = df['Data da Notificação'].str[:4]

In [20]:
# Convertendo de volta para o formato correto
df['Data_Completa_Notificacao'] = pd.to_datetime(df['Data da Notificação'], errors='coerce')

# Tratando múltiplas informações nas colunas vacina e dose

As colunas vacina e dose apresentam mais de uma informação por entrada.
É de extrema importância dividir esses informações em colunas adjacentes, visando facilitar a busca, leitura e processamento dos dados nessas colunas.

In [ ]:
# separando as vacinas pelo separador / e criando novas colunas para cada vacina
vacinas_separadas = df['Imunobiológico (vacina)'].str.split('/', expand=True).add_prefix('Vacina_')

In [ ]:
# separando as doses pelo separador / e criando novas colunas para cada dose
doses_separadas = df['Dose'].str.split('/', expand=True).add_prefix('Dose_')

In [29]:
for col in vacinas_separadas.columns:
    # Remove espaços em branco nas pontas
    vacinas_separadas[col] = vacinas_separadas[col].str.strip()
    # Remove o padrão de numeração inicial (ex: "1: ") se ele existir
    vacinas_separadas[col] = vacinas_separadas[col].str.replace(r'^\d+:\s*', '', regex=True)

for col in doses_separadas.columns:
    doses_separadas[col] = doses_separadas[col].str.strip()
    doses_separadas[col] = doses_separadas[col].str.replace(r'^\d+:\s*', '', regex=True)

In [ ]:
df = pd.concat([df, vacinas_separadas, doses_separadas], axis=1) # juntando as novas colunas de vacinas e doses ao DataFrame original

# Limpeza e Padronização dos Eventos e Desfechos

Vamos aplicar uma limpeza em lote nessas colunas de uma vez só, usando expressões regulares (regex) para remover os prefixos numéricos e os espaços em branco que sobram nas pontas.

In [34]:
colunas_para_limpar = [
    'Tipo de Evento', 
    'Reação / evento adverso', 
    'Classificação de gravidade', 
    'Desfecho (evolução do caso)'
]

for col in colunas_para_limpar:
    # Garantir que a coluna seja tratada como texto (string)
    df[col] = df[col].astype(str)
    
    # Remover o padrão numérico do início 
    df[col] = df[col].str.replace(r'^\d+:\s*', '', regex=True)
    
    # Remover espaços em branco inúteis no início e no final do texto
    df[col] = df[col].str.strip()
    
    # Substituir textos que viraram "nan" (por estarem vazios antes) por um valor nulo real (None)
    df[col] = df[col].replace(['nan', 'NaN', '0'], None)

In [35]:
print(df[colunas_para_limpar].head())

       Tipo de Evento Reação / evento adverso Classificação de gravidade  \
0  Erro de Imunização         Dose inadequada                  Não grave   
1  Erro de Imunização         Dose inadequada                  Não grave   
2  Erro de Imunização         Dose inadequada                  Não grave   
3  Erro de Imunização         Dose inadequada                  Não grave   
4  Erro de Imunização         Dose inadequada                  Não grave   

  Desfecho (evolução do caso)  
0           Cura sem sequelas  
1           Cura sem sequelas  
2           Cura sem sequelas  
3           Cura sem sequelas  
4           Cura sem sequelas  


# Salvando localmente

Trecho simples para salvamendo local do novo DataFrame em um xlsx.

In [38]:
nome_do_arquivo = '../data/dados_tratados_parcial.xlsx'

df.to_excel(nome_do_arquivo, index=False)